# Translate metadata ready for merging with main metadata
This notebook processes metadata from a study where:
- Location comes from the "Country" column
- Date comes from the "Year" column
- Isolation source comes from the existing "isolation_source" column (no translation needed)

In [12]:
# HARMONIZED MAPPING CODE FOR KLEBSIELLA PROJECT
# Copy this entire code block into a new cell in your notebook

# Import the harmonized mapping functions
import pandas as pd
import sys
import os
sys.path.append(os.getcwd())  # Add current directory to path
from bac_metadata.pp.metadata_ENA_study import (setup_standardized_columns, debug_sample_matching,
                               map_metadata, display_mapping_results, save_results)

# ===== VARIABLE DEFINITIONS =====
# Study accession
study_accession = "PRJEB5065"
#['PRJEB1271' 'PRJEB5065' 'PRJEB7657' 'PRJEB2111']

other_study_accession = ""
all_accessions = [study_accession, other_study_accession]
if other_study_accession != "":     
    string_of_accessions = f"{study_accession}_{other_study_accession}"
else:
    string_of_accessions = study_accession
# Change for different studies
data_dir = "/Users/davidabelson/Library/CloudStorage/OneDrive-UniversityofCambridge/Aaron Weimann's files - project_k/data/raw/metadata"
project_k_study_dir = f"{data_dir}/study_level_metadata/ENA_projects/{string_of_accessions}"
#project_k_study_dir = "/Users/davidabelson/Library/CloudStorage/OneDrive-UniversityofCambridge/Aaron Weimann's files - project_k/data/raw/metadata/study_level_metadata/ENA_projects/PRJEB6891, PRJNA351909"

# ===== STEP 2: ENA DATA ON PROJECT NUMBER =====
# This section loads the ENA metadata and prepares it for updating

# STEP 2.1: Load the ENA metadata file
#Initial data release
ena_file = f"{data_dir}/ena_metadata_klebsiella_with_header_filtered.tsv"
ena_data_initial = pd.read_csv(ena_file, sep="\t", low_memory=False)  # Added low_memory=False to avoid warnings
#Incremental release
ena_file = f"{data_dir}/ena_metadata_klebsiella_with_header_filtered_r02_format.20240801.tsv"
ena_data_incremental = pd.read_csv(ena_file, sep="\t", low_memory=False)  # Added low_memory=False to avoid warnings

# Bakrep extra ENA metadata
ena_file = f"{data_dir}/bakrep_klebsiella_genus_extra_ena_metadata.tsv"
ena_data_bakrep = pd.read_csv(ena_file, sep="\t", low_memory=False)  # Added low_memory=False to avoid warnings

# Append together the incremental and initial data releases
ena_data = pd.concat([ena_data_initial, ena_data_incremental, ena_data_bakrep])    
# Total number of samples in the ENA data before filtering
print(f"Total number of samples in the ENA data before filtering: {len(ena_data)}")

# STEP 2: report on current ENA metadata completeness
# Match using both study_accession and other_study_accession values in the same column

ena_study_subset = ena_data[ena_data["study_accession"].isin(all_accessions)]
# Number where sceinific name includes words "Klebsiella"
print(f"Number of samples where scientific name includes words 'Klebsiella': {len(ena_study_subset[ena_study_subset['scientific_name'].str.contains('Klebsiella')])}")
from bac_metadata.pp.metadata_curation import report_ena_data_completeness
report_ena_data_completeness(ena_study_subset, filter_n = 0, report_columns=True, display_n=3)

Total number of samples in the ENA data before filtering: 92928
Number of samples where scientific name includes words 'Klebsiella': 379
The total number of samples in the ENA data file is 383
The total number of study accessions in the ENA data file is 1

After filtering (keeping studies with > 0 unique samples):
  Number of samples: 383
  Number of study accessions: 1

The first 3 study accessions in the ENA data file are: ['PRJEB5065']

Columns completeness summary:


,column,n_filled,n_missing,unique_values
0,host,299,84,1
1,country,299,84,2
2,collection_date,299,84,12
3,isolation_source,299,84,2
4,dev_stage,0,383,0



Detailed column reports:

Column: host
  Present: 299
  Missing: 84
  Unique values (1 total):
    - Homo sapiens: 299

Column: country
  Present: 299
  Missing: 84
  Unique values (2 total):
    - UK: 297
    - United Kingdom: 2

Column: collection_date
  Present: 299
  Missing: 84
  Unique values (12 total):
    - 2006-01-01: 42
    - 2004-01-01: 39
    - 2005-01-01: 36
    ... and 9 more

Column: isolation_source
  Present: 299
  Missing: 84
  Unique values (2 total):
    - Blood: 297
    - blood: 2

Column: dev_stage
  Present: 0
  Missing: 383
  Unique values (0 total):


(               host         country collection_date isolation_source  \
 57766           NaN             NaN             NaN              NaN   
 57767  Homo sapiens  United Kingdom      1800-01-01            blood   
 57768           NaN             NaN             NaN              NaN   
 57769  Homo sapiens  United Kingdom      1800-01-01            blood   
 57770           NaN             NaN             NaN              NaN   
 ...             ...             ...             ...              ...   
 58164           NaN             NaN             NaN              NaN   
 348             NaN             NaN             NaN              NaN   
 349             NaN             NaN             NaN              NaN   
 350    Homo sapiens              UK      2003-01-01            Blood   
 351    Homo sapiens              UK      2008-01-01            Blood   
 
        dev_stage  
 57766        NaN  
 57767        NaN  
 57768        NaN  
 57769        NaN  
 57770        NaN  
 .

In [ ]:
display(ena_study_subset.head())

In [ ]:
# If need  names of columns in ENA file
# View col names including "study"
study_cols = [col for col in ena_data.columns if "study" in col.lower()]
print(f"study_cols in ENA file: {study_cols}")

sample_cols = [col for col in ena_data.columns if "sample" in col.lower()]
print(f"sample_cols in ENA file: {sample_cols}")

ena_study_subset = ena_data[ena_data["study_accession"] == study_accession]
print(f"ENA subset for {study_accession}: {len(ena_study_subset)} samples")

# STEP 2.3: Further filter to only samples that match our donor_data
# - This creates a subset where the run_accession in ena_data matches the "Run accession" in donor_data
# - recipient_data is our working dataset that we'll use to update the main ENA data
# ENA data "sample_accession" columns
sample_accession_cols = [col for col in ena_data.columns if "sample_accession" in col.lower()]
print(f"ENA file sample_accession_cols: {sample_accession_cols}")

run_accession_cols = [col for col in ena_data.columns if "run_accession" in col.lower()]
print(f"ENA file run_accession_cols: {run_accession_cols}")

In [3]:
# STEP 3:  If non-complete, then look at study file metadata for missing values

study_metadata_file = f"{project_k_study_dir}/data.csv"
study_metadata = pd.read_csv(study_metadata_file)

# ===== STEP 1: STANDARDIZE STUDY DATA =====
print("="*60)
print(f"STUDY METADATA FILE, study accession: {study_accession}")
print("="*60)

# Print the total number of samples in the study
print(f"Total number of samples in the study: {len(study_metadata)}")

species_col = "SPECIES"
# Filter study data to Klebsiella only if possible
if species_col in study_metadata.columns:
    study_metadata_klebsiella = study_metadata[study_metadata[species_col].str.contains("Klebsiella", na=False)]
    print(f"Study data filtered to Klebsiella: {len(study_metadata_klebsiella)}/{len(study_metadata)} samples")
else:
    print("No species column found, using all study data")
    study_metadata_klebsiella = study_metadata

print(f"The total nuber of klebsiella samples in the study metadata file is {len(study_metadata_klebsiella)}")
# Filter for project accession in case the study file has more than one project number
#study_metadata_klebsiella = study_metadata_klebsiella[study_metadata_klebsiella["PROJECT ACCESSION"] == study_accession]

#display(study_data_for_mapping.head())
# columns with name 'sample' in the name
sample_cols = [col for col in study_metadata_klebsiella.columns if "sample" in col.lower()]
print(f"Columns with 'sample' in the name: {sample_cols}")
run_accession_cols = [col for col in study_metadata_klebsiella.columns if "run" in col.lower()]
print(f"Columns with 'run' in the name: {run_accession_cols}")
run_accession_cols = [col for col in study_metadata_klebsiella.columns if "accession" in col.lower()]
print(f"Columns with 'run' in the name: {run_accession_cols}")
print(f"All columns: {study_metadata_klebsiella.columns.tolist()}")

print(f"Opened {study_accession} study metadata file: {study_metadata_file}")



STUDY METADATA FILE, study accession: PRJEB1271
Total number of samples in the study: 162
No species column found, using all study data
The total nuber of klebsiella samples in the study metadata file is 162
Columns with 'sample' in the name: ['SampleSite', 'sample ID', 'sample_accession']
Columns with 'run' in the name: ['runi ID']
Columns with 'run' in the name: ['sample_accession']
All columns: ['lane ID', 'runi ID', 'strain', 'strain ID', 'Month', 'SampleSite', 'species', 'sample ID', 'sample_accession', 'year', 'two-year block', 'ST', 'aminoglycoside.AAC.3..IIa', 'aminoglycoside.AAC.6...Ib.cr', 'aminoglycoside.AAC.6...IIa', 'aminoglycoside.AAC.6...IIc', 'aminoglycoside.aadA1', 'aminoglycoside.aadA16', 'aminoglycoside.aadA2', 'aminoglycoside.aadA5', 'aminoglycoside.ANT.2....Ia', 'aminoglycoside.APH.3...Ia', 'aminoglycoside.armA', 'aminoglycoside.Sat.1', 'aminoglycoside.strA.APH.3....Ib.', 'aminoglycoside.strB.APH.6..Id.', 'beta.lact.ampC.', 'beta.lact.ampH', 'beta.lact.CARB.12', 'b

In [ ]:
# # STEP 2.7: 
# Find columns in the study metadata file that have "host", "location", "country", "collection_date", "isolation_source" in the name
host_cols = [col for col in study_metadata_klebsiella.columns if "host" in col.lower()]
print(f"Columns with 'host' in the name: {host_cols}")
location_cols = [col for col in study_metadata_klebsiella.columns if "location" in col.lower()]
print(f"Columns with 'location' in the name: {location_cols}")
country_cols = [col for col in study_metadata_klebsiella.columns if "country" in col.lower()]
print(f"Columns with 'country' in the name: {country_cols}")
date_cols = [col for col in study_metadata_klebsiella.columns if "date" in col.lower()]
print(f"Columns with 'date' in the name: {date_cols}")
source_cols = [col for col in study_metadata_klebsiella.columns if "source" in col.lower()]
print(f"Columns with 'source' in the name: {source_cols}")

print(f"List of all columns: {study_metadata_klebsiella.columns.tolist()}")


In [4]:
display(study_metadata_klebsiella.head())

,lane ID,runi ID,strain,strain ID,Month,SampleSite,species,sample ID,sample_accession,year,...,tetracycline.tetA,tetracycline.tetD,tetracycline.tetR,trimethoprim.dfrA.,physical fluoroquinolone,physical beta-lactam,physical aminogycosides,physical trimotheprim,K-type,O-type
0,9221_7#92,ERR257689,Kpn79,AKPRH094188,28,BLOOD,K. quasipneumoniae subsp. similipneumoniae,ERS217771,SAMEA1713074,2008,...,no,no,no,no,R,S,S,R,KL6,O12
1,9221_7#12,ERR257609,Kpn9,AKPRH080251,20,BLOOD,Klebsiella pneumoniae,ERS217691,SAMEA1713061,2007,...,no,yes,no,yes,R,R,R,R,KL17,O1v1
2,9221_7#13,ERR257610,Kpn10,AKPRH097254,30,BLOOD,Klebsiella pneumoniae,ERS217692,SAMEA1713011,2008,...,no,partial,no,yes,R,R,S,R,KL17,O1v1
3,9221_7#31,ERR257628,Kpn26,AKPRH100898,47,ABDOMEN,Klebsiella pneumoniae,ERS217710,SAMEA1712947,2009,...,yes,no,yes,no,R,S,R,S,KL10,O1v1
4,9221_7#32,ERR257629,Kpn27,AKPRH111435,55,ABDOMEN,Klebsiella pneumoniae,ERS217711,SAMEA1712968,2010,...,yes,no,yes,yes,R,R,R,R,KL112,O1v1


In [6]:
# ===== AUTOMATIC KEY COLUMN DETECTION =====
# This code automatically finds which columns match between ENA and study metadata

def find_matching_columns(ena_df, study_df, ena_candidate_cols, study_candidate_cols, sample_size=5):
    """
    Find which columns from ena_df and study_df have matching values.
    
    Parameters:
    - ena_df: The ENA subset dataframe
    - study_df: The study metadata dataframe
    - ena_candidate_cols: List of column names in ENA data to check
    - study_candidate_cols: List of column names in study data to check
    - sample_size: Number of values to sample from study data for matching
    
    Returns:
    - tuple: (best_ena_col, best_study_col, match_count) or (None, None, 0) if no match found
    """
    best_match = (None, None, 0)
    
    print("\n" + "="*60)
    print("SEARCHING FOR MATCHING KEY COLUMNS")
    print("="*60)
    
    for study_col in study_candidate_cols:
        if study_col not in study_df.columns:
            continue
            
        # Get first few non-null values from study column
        sample_values = study_df[study_col].dropna().head(sample_size).tolist()
        
        if not sample_values:
            continue
            
        print(f"\nTrying study column '{study_col}' with sample values: {sample_values[:3]}...")
        
        for ena_col in ena_candidate_cols:
            if ena_col not in ena_df.columns:
                continue
                
            # Count how many of the sample values exist in the ENA column
            matches_in_sample = sum(val in ena_df[ena_col].values for val in sample_values)
            
            # Count total potential matches
            total_matches = ena_df[ena_col].isin(study_df[study_col]).sum()
            
            if total_matches > 0:
                print(f"  ✓ ENA column '{ena_col}': {matches_in_sample}/{len(sample_values)} sample matches, {total_matches} total matches")
                
                if total_matches > best_match[2]:
                    best_match = (ena_col, study_col, total_matches)
            
    if best_match[0] is not None:
        print(f"\n{'='*60}")
        print(f"BEST MATCH FOUND:")
        print(f"  ENA column: '{best_match[0]}'")
        print(f"  Study column: '{best_match[1]}'")
        print(f"  Total matches: {best_match[2]}")
        print(f"{'='*60}\n")
    else:
        print(f"\n{'='*60}")
        print(f"WARNING: No matching columns found!")
        print(f"{'='*60}\n")
    
    return best_match[0], best_match[1], best_match[2]

# Define candidate columns to check
ena_candidate_cols = [
    'sample_accession', 
    'secondary_sample_accession', 
    'run_accession',
    'sample_alias',
    'isolate',
    'experiment_alias'
]

study_candidate_cols = [
    'ID',
    'Alternative sample id',
    'Biosample',
    'Run Accession', 
    'Run accession',
    'Sample Accession',
    'Sample accession',
    'Reads Accession',
    'sample_accession',
    'run_accession',
    'RunID',
    'Reads accession (Illumina)',
    'Reads accession (Ion Torrent)',
    'Reads accession (PacBio)',
    'sample_title',
    'Isolate',
    'experimental_alias',
    'runi ID',
    'strain ID'

]

# Automatically find the best matching columns
ena_key_col, study_key_col, match_count = find_matching_columns(
    ena_study_subset, 
    study_metadata_klebsiella,
    ena_candidate_cols,
    study_candidate_cols,
    sample_size=10
)

# Verify we found a match
if ena_key_col is None or study_key_col is None:
    print("❌ ERROR: Could not automatically detect matching columns!")
    print("Please manually specify ena_key_col and study_key_col")
else:
    print(f"✓ Using ENA key column: '{ena_key_col}'")
    print(f"✓ Using Study key column: '{study_key_col}'")
    print(f"✓ Number of matching rows: {match_count}")


SEARCHING FOR MATCHING KEY COLUMNS

Trying study column 'sample_accession' with sample values: ['SAMEA1713074', 'SAMEA1713061', 'SAMEA1713011']...
  ✓ ENA column 'sample_accession': 10/10 sample matches, 162 total matches

Trying study column 'runi ID' with sample values: ['ERR257689', 'ERR257609', 'ERR257610']...
  ✓ ENA column 'run_accession': 10/10 sample matches, 162 total matches

Trying study column 'strain ID' with sample values: ['AKPRH094188', 'AKPRH080251', 'AKPRH097254']...

BEST MATCH FOUND:
  ENA column: 'sample_accession'
  Study column: 'sample_accession'
  Total matches: 162

✓ Using ENA key column: 'sample_accession'
✓ Using Study key column: 'sample_accession'
✓ Number of matching rows: 162


In [ ]:
# Make Country = f"{Country}: {Regions}"
study_metadata_klebsiella.loc[:,"Country"] = study_metadata_klebsiella.loc[:,"Country"] + ": " + study_metadata_klebsiella.loc[:,"Regions"]


In [5]:
# Define which columns to use as keys for matching between datasets
ena_key_col = "sample_accession"  # Column in ena_study_subset to use as key
study_key_col = "sample_accession"  # Column in study_metadata_klebsiella to use as key

# print how many rows in study_metadata_klebsiella have a value in the study_key_col
print(f"Number of rows in study_metadata_klebsiella: {len(study_metadata_klebsiella)}")
print(f"Number of rows in study_metadata_klebsiella with a value in the study_key_col: {study_metadata_klebsiella[study_key_col].notna().sum()}")
# Number of rows which match between the two datasets
print(f"Number of ENA rows which match a key in Klebsiella metatdata using the two keys: {ena_study_subset[ena_key_col].isin(study_metadata_klebsiella[study_key_col]).sum()}")


# "name_in_ENA": "name_in_study_metadata"
columns_to_map = {
    "host": None,
    "location": None,
    "country": None,
    "collection_date": "year",
    "isolation_source": "SampleSite",
    #'dev_stage': 'Patient age'
}

# Wipe the isolation source column
ena_study_subset.loc[:,"isolation_source"] = None
ena_study_subset.loc[:,"collection_date"] = None

# ===== STEP 3: UPDATE ORIGINAL ENA DATA WITH MODIFIED VALUES =====
# This section updates the ena_study_subset with values from study_metadata_klebsiella, but ONLY where values are missing

# Loop through each column we want to check and potentially update
for ena_column, study_column in columns_to_map.items():
    # Skip columns that don't have a mapping in the study metadata
    if study_column is None:
        print(f"Skipping column '{ena_column}' as it has no mapping in study metadata")
        continue
    
    # STEP 3.1: Create a mapping dictionary from study key column to the study column values
    # - Filter study_metadata to only rows where this column has a value (not NA)
    # - Create a dictionary with study_key_col as keys and column values as values
    value_map = {}
    valid_rows = study_metadata_klebsiella[study_metadata_klebsiella[study_column].notna()]
    print(f"Valid rows for {study_column}: {len(valid_rows)}")
    for _, row in valid_rows.iterrows():
        if pd.notna(row[study_key_col]):
            value_map[row[study_key_col]] = row[study_column]
    
    # STEP 3.2: Update ena_study_subset where values are missing
    # - Create a mask that identifies rows where:
    #   1. The ena_key_col is in our value_map keys (we have a value for this sample)
    #   2. AND the column value is currently NA in ena_study_subset
    mask = (ena_study_subset[ena_key_col].isin(value_map.keys())) & (ena_study_subset[ena_column].isna())
    
    # STEP 3.3: Apply the update using the mask
    # - For rows matching our mask, look up the ena_key_col in our value_map
    # - Replace the NA values with the values from our mapping
    ena_study_subset.loc[mask, ena_column] = ena_study_subset.loc[mask, ena_key_col].map(value_map)
    
    # Print summary of updates for this column
    num_updated = mask.sum()
    print(f"Updated {num_updated} missing values in column '{ena_column}' using data from '{study_column}'")


Number of rows in study_metadata_klebsiella: 162
Number of rows in study_metadata_klebsiella with a value in the study_key_col: 162
Number of ENA rows which match a key in Klebsiella metatdata using the two keys: 162
Skipping column 'host' as it has no mapping in study metadata
Skipping column 'location' as it has no mapping in study metadata
Skipping column 'country' as it has no mapping in study metadata
Valid rows for year: 162
Updated 162 missing values in column 'collection_date' using data from 'year'
Valid rows for SampleSite: 162
Updated 162 missing values in column 'isolation_source' using data from 'SampleSite'


In [ ]:
#ena_study_subset.loc[:,"country"] = "Nigeria"
#ena_study_subset.loc[:,"collection_date"] = "2020/06/01"
# If host is na then hard code to "clinical environment"    
#ena_study_subset.loc[ena_study_subset.loc[:,"host"].isna(), "host"] = "human"
#ena_study_subset.loc[ena_study_subset.loc[:,"isolation_source"].isna(), "isolation_source"] = "wastewater"
#ena_study_subset.loc[:, "country"] = "Netherlands: Groningen"
#ena_study_subset.loc[:,"isolation_source"] = "wastewater and faeces"
# If isoltation source = "clinical" then host = "human" and isolation source = None
#ena_study_subset.loc[:,"dev_stage"] = "adult"

# #ena_study_subset.loc[:,"isolation_source"] = None
# ena_study_subset.loc[:,"country"] = "Philippines"
#ena_study_subset.loc[:,"dev_stage"] = "adult"

# If collection date is na then hard code to 2015-01-01
#if ena_study_subset.loc[:,"collection_date"].isna().any():
#ena_study_subset.loc[:,"collection_date"] = "2016/06/30"
# ena_study_subset.loc[:,"isolation_source"] = "blood"
#ena_study_subset.loc[:,"location"] = ena_study_subset.loc[:,"country"]
#ena_study_subset.loc[:,"country"] = "Netherlands"
# Change any isolation source which is 'swab' to 'rectal swab'
#ena_study_subset.loc[:, 'isolation_source'] = "rectal swab"
#ena_study_subset.loc[:, 'dev_stage'] = "adult"


In [9]:
report_ena_data_completeness(ena_study_subset, filter_n = 0, report_columns=True, display_n=3)

The total number of samples in the ENA data file is 224
The total number of study accessions in the ENA data file is 1

After filtering (keeping studies with > 0 unique samples):
  Number of samples: 224
  Number of study accessions: 1

The first 3 study accessions in the ENA data file are: ['PRJEB1271']

Columns completeness summary:


,column,n_filled,n_missing,unique_values
0,host,165,59,1
1,country,165,59,1
2,collection_date,162,62,7
3,isolation_source,162,62,14
4,dev_stage,0,224,0



Detailed column reports:

Column: host
  Present: 165
  Missing: 59
  Unique values (1 total):
    - human: 165

Column: country
  Present: 165
  Missing: 59
  Unique values (1 total):
    - UK: 165

Column: collection_date
  Present: 162
  Missing: 62
  Unique values (7 total):
    - 2011: 50
    - 2006: 29
    - 2009: 22
    ... and 4 more

Column: isolation_source
  Present: 162
  Missing: 62
  Unique values (14 total):
    - BLOOD: 106
    - URINE: 14
    - SPUTUM: 13
    ... and 11 more

Column: dev_stage
  Present: 0
  Missing: 224
  Unique values (0 total):


(        host country collection_date isolation_source  dev_stage
 8176   human      UK            2007            BLOOD        NaN
 56943  human      UK            2006            BLOOD        NaN
 56944  human      UK            2009            BLOOD        NaN
 56945  human      UK            2010            BLOOD        NaN
 56946  human      UK            2007            BLOOD        NaN
 ...      ...     ...             ...              ...        ...
 57620    NaN     NaN            None             None        NaN
 57621  human      UK            2007            BLOOD        NaN
 344      NaN     NaN            None             None        NaN
 345      NaN     NaN            None             None        NaN
 346      NaN     NaN            None             None        NaN
 
 [224 rows x 5 columns],
              column  n_filled  n_missing  unique_values
 0              host       165         59              1
 1           country       165         59              1
 2   colle

In [8]:
# ===== STEP 5: SAVE STUDY-SPECIFIC DATA FOR MERGING =====
# This saves a copy of just this study's data for merging with other studies

# Make sure the directory exists
import os
# # Save the study-specific data that we've processed locally
# ena_study_subset.to_csv(f"ready_to_merge/{study_accession}_ready_to_merge_with_main_meta_corrected.csv", index=False)
# print(f"Study data saved to ./ready_to_merge/{study_accession}_ready_to_merge_with_main_meta_corrected.csv")

# Save the project k drive
# mkdir
ena_file_dir = project_k_study_dir
os.makedirs(ena_file_dir, exist_ok=True)
ena_file_path = f"{ena_file_dir}/{study_accession}_ready_to_merge_with_main_meta_corrected.csv"
ena_study_subset.to_csv(ena_file_path, index=False)
print(f"ENA metadata saved to {ena_file_path}")

ENA metadata saved to /Users/davidabelson/Library/CloudStorage/OneDrive-UniversityofCambridge/Aaron Weimann's files - project_k/data/raw/metadata/study_level_metadata/ENA_projects/PRJEB1271/PRJEB1271_ready_to_merge_with_main_meta_corrected.csv


In [10]:
# Reload the file
data_check = pd.read_csv(ena_file_path)
# Loaded data from the file path:
print(f"Loaded data from the file path: {ena_file_path}")

# Data completeness
report_ena_data_completeness(data_check, filter_n = 0, report_columns=True, display_n=3)


Loaded data from the file path: /Users/davidabelson/Library/CloudStorage/OneDrive-UniversityofCambridge/Aaron Weimann's files - project_k/data/raw/metadata/study_level_metadata/ENA_projects/PRJEB1271/PRJEB1271_ready_to_merge_with_main_meta_corrected.csv
The total number of samples in the ENA data file is 224
The total number of study accessions in the ENA data file is 1

After filtering (keeping studies with > 0 unique samples):
  Number of samples: 224
  Number of study accessions: 1

The first 3 study accessions in the ENA data file are: ['PRJEB1271']

Columns completeness summary:


,column,n_filled,n_missing,unique_values
0,host,165,59,1
1,country,165,59,1
2,collection_date,162,62,7
3,isolation_source,162,62,14
4,dev_stage,0,224,0



Detailed column reports:

Column: host
  Present: 165
  Missing: 59
  Unique values (1 total):
    - human: 165

Column: country
  Present: 165
  Missing: 59
  Unique values (1 total):
    - UK: 165

Column: collection_date
  Present: 162
  Missing: 62
  Unique values (7 total):
    - 2011.0: 50
    - 2006.0: 29
    - 2009.0: 22
    ... and 4 more

Column: isolation_source
  Present: 162
  Missing: 62
  Unique values (14 total):
    - BLOOD: 106
    - URINE: 14
    - SPUTUM: 13
    ... and 11 more

Column: dev_stage
  Present: 0
  Missing: 224
  Unique values (0 total):


(      host country  collection_date isolation_source  dev_stage
 0    human      UK           2007.0            BLOOD        NaN
 1    human      UK           2006.0            BLOOD        NaN
 2    human      UK           2009.0            BLOOD        NaN
 3    human      UK           2010.0            BLOOD        NaN
 4    human      UK           2007.0            BLOOD        NaN
 ..     ...     ...              ...              ...        ...
 219    NaN     NaN              NaN              NaN        NaN
 220  human      UK           2007.0            BLOOD        NaN
 221    NaN     NaN              NaN              NaN        NaN
 222    NaN     NaN              NaN              NaN        NaN
 223    NaN     NaN              NaN              NaN        NaN
 
 [224 rows x 5 columns],
              column  n_filled  n_missing  unique_values
 0              host       165         59              1
 1           country       165         59              1
 2   collection_date  